# MAGFiLO Mask2Former v1

Add Data → `filament-segmentation-2026`. Clone this repo into `/kaggle/working` or Add Dataset (zip without `.venv`). GPU on. Smoke first, then train.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient

    sec = UserSecretsClient()
    for name in ("GIT_URL", "GITHUB_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name)
        except Exception:
            pass
except Exception:
    pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
url = os.environ.get("GIT_URL", "")
if url and not (REPO / "run.py").exists():
    token = os.environ.get("GITHUB_TOKEN", "")
    if token and url.startswith("https://"):
        url = url.replace("https://", f"https://{token}@", 1)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(REPO)])

root = (
    REPO
    if (REPO / "run.py").exists()
    else next(p for p in Path("/kaggle/input").rglob("run.py")).parent
)
sys.path.insert(0, str(root))
os.chdir(root)
print(root)
!pip -q install transformers accelerate safetensors
from src.data import TRAIN_ANN, TRAIN_IMAGES

assert TRAIN_ANN.exists() and TRAIN_IMAGES.exists(), TRAIN_ANN

In [ ]:
!python run.py check
!python run.py train --smoke
# after smoke:
# !python run.py train --size 512 --epochs 8
# !python run.py infer --submit